# Conformal Factuality — Distribution-Free Correctness Guarantees for Generation

The narrative companion to `conformal_factuality.py`, the canonical reference that **owns every number**.
This notebook imports that module and walks the topic movement by movement; it never redefines the math.

The eval layer's through-line — *a metric is an estimator* — reaches its terminus. The prerequisite
(`llm-as-judge-ragas`) produced a calibrated, debiased per-claim **judge confidence**. Here we turn it into
a **distribution-free correctness guarantee**: conformal prediction converts *any* nonconformity score into
finite-sample coverage under **exchangeability alone** — no model of when the LLM hallucinates. We run split
conformal (a recall guarantee), then **Conformal Risk Control** (the false-claim-rate guarantee that
actually matters), then break exchangeability with covariate shift and repair it with weighted conformal.

In [ ]:
import pathlib, sys

# Path-robust import: find the module regardless of the kernel's working directory. The module itself adds
# every ancestor notebook dir (the retrieval subtree + both eval prereqs) to sys.path on import.
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "conformal-factuality",
              pathlib.Path("notebooks/conformal-factuality")):
    if (_cand / "conformal_factuality.py").exists():
        sys.path.insert(0, str(_cand.resolve()))
        break

import numpy as np
import conformal_factuality as CF
from conformal_factuality import (
    corpus, pooled_confidence, calibrated_confidence, split_masks,
    split_conformal_threshold, confidence_threshold, back_off_retained,
    false_claim_loss, fraction_loss, conformal_risk_control_threshold,
    weighted_conformal_threshold, mc_realized_coverage, risk_coverage_curve,
    covariate_shift_curve, worked_answer, judge_calibration_quality,
    oracle_faithfulness, precision_at_k, JUDGE, WORKED_LEG, ALPHA, K, LEG_NAMES,
)

c = corpus()
print(f"corpus: {c['n_queries']} queries, {c['n_docs']} docs, k={K} claims/answer; leg={WORKED_LEG}")

## Movement 0 — the score: a calibrated judge confidence

Each (query, claim) pair carries the judge's stated probability that the claim is *supported*. The judge
here is deliberately **lenient** (over-endorses): informative (AUC ≈ 0.90) but its faithful and unfaithful
confidences **overlap**, so unsupported claims leak through at high confidence. The prereq's calibration
suite (Platt) recalibrates the raw confidence into a probability. A key fact: recalibration is **strictly
monotone**, so it leaves the ranking — and the AUC — *exactly* unchanged. It buys **efficiency**, not
validity.

In [ ]:
cal = judge_calibration_quality(c, WORKED_LEG, JUDGE)
print(f"judge confidence as a probability:")
print(f"  ECE raw   = {cal['ece_raw']:.4f}   ECE Platt = {cal['ece_platt']:.4f}   (recalibration lowers it)")
print(f"  AUC raw   = {cal['auc_raw']:.4f}   AUC Platt = {cal['auc_platt']:.4f}   (monotone map: AUC unchanged)")

## Movement 1 — split conformal and the recall guarantee

The nonconformity score is `s = 1 − c̃` (low score = confident-faithful). We hold out a calibration set,
restrict to the **truly-faithful** claims, and take the `⌈(1−α)(n+1)⌉`-th smallest score as the threshold
`τ̂` (formalML's Theorem 1). Retain a claim iff its confidence clears `1 − τ̂`. The guarantee: a genuinely
faithful claim is retained with probability ≥ 1 − α — a **recall** guarantee. We verify it by Monte-Carlo
resplitting: realized coverage tracks 1 − α.

In [ ]:
print(f"{'alpha':>6} {'target':>8} {'realized':>10} {'std':>8}")
for a in (0.05, 0.10, 0.20):
    mc = mc_realized_coverage(c, WORKED_LEG, JUDGE, a)
    print(f"{a:6.2f} {mc['target']:8.2f} {mc['mean']:10.4f} {mc['std']:8.4f}")

## Movement 2 — recall is not precision: Conformal Risk Control

The recall guarantee says nothing about **hallucination leakage**. With a lenient judge, the false-claim
rate among retained claims runs *uncontrolled* — far above α. To control it we need a guarantee on a
**monotone loss**: the per-slot false-claim rate `L(λ) = (1/k)·#{retained ∧ unfaithful}`. Its denominator
is the *fixed* slot count k, so it is non-increasing in the confidence cut — exactly CRC's hypothesis. The
naive *fraction*-of-retained loss divides by the *shrinking* retained count and is **not** monotone.

In [ ]:
# the naive fraction loss is NON-monotone -- why CRC needs the fixed-denominator loss
p = np.array([0.55, 0.54, 0.53, 0.52, 0.51, 0.505, 0.504, 0.503, 0.502, 0.501])
y = np.array([0, 1, 1, 1, 1, 1, 1, 1, 1, 1])
print(f"fraction loss: cut 0.50 -> {fraction_loss(p,y,0.50):.3f}   cut 0.545 -> {fraction_loss(p,y,0.545):.3f}  (ROSE)")
print(f"monotone loss: cut 0.50 -> {false_claim_loss(p,y,0.50):.3f}   cut 0.545 -> {false_claim_loss(p,y,0.545):.3f}  (fell)")
print()

# the headline frontier: split conformal's recall lets false claims through; CRC controls them
rows = risk_coverage_curve(c, WORKED_LEG, JUDGE)
print(f"{'alpha':>6} {'split_recall':>13} {'split_FALSE':>12} {'crc_FALSE':>11} {'crc_retention':>14}")
for r in rows:
    print(f"{r['alpha']:6.2f} {r['split_recall']:13.3f} {r['split_false']:12.3f} {r['crc_false']:11.3f} {r['crc_retention']:14.3f}")

The `split_FALSE` column is the recall guarantee's blind spot — at α = 0.02 nearly a quarter of retained
slots are hallucinations. The `crc_FALSE` column respects the y = α line (the lone small overshoot is the
honest face of CRC: it controls the *expectation* over calibration draws, not a single realization).

## Movement 3 — drift breaks exchangeability; weighted conformal repairs it

Every guarantee above is **marginal** and rests on **exchangeability**. A deployment that over-represents
low-verbosity claims is a **covariate shift** (a known likelihood ratio `w(x) = exp(−βx)`): the threshold
calibrated on the past under-covers the present. Weighted split-conformal (Tibshirani et al. 2019) reweights
the calibration scores by `w` and restores coverage at the shifted distribution.

In [ ]:
rows = covariate_shift_curve(c, WORKED_LEG, JUDGE, ALPHA)
print(f"{'beta':>6} {'target':>8} {'split_cov':>10} {'weighted_cov':>13}")
for r in rows:
    flag = "  <-- split broken" if r['split'] < r['target'] - 0.02 else ""
    print(f"{r['beta']:6.2f} {r['target']:8.2f} {r['split']:10.3f} {r['weighted']:13.3f}{flag}")

## The collapse anchor — a perfect judge recovers precision@k exactly

A perfect judge (sensitivity = specificity = 1) separates faithful from unfaithful confidence cleanly. The
back-off retained set is then *exactly* the faithful claims, so the per-query retained fraction equals the
imported `oracle_faithfulness` == `precision_at_k` — to machine precision. This anchors the whole
construction onto the metric the entire eval layer was built on.

In [ ]:
from conformal_factuality import JUDGE_PERFECT
leg = WORKED_LEG
conf, yv, _ = pooled_confidence(c, leg, JUDGE_PERFECT)
P, Y = conf.reshape(c['n_queries'], K), yv.reshape(c['n_queries'], K)
oracle = oracle_faithfulness(c, leg)
gaps = []
for q in range(c['n_queries']):
    retained = back_off_retained(P[q], 0.5)['n_retained'] / K
    gaps.append(abs(retained - oracle[q]))
print(f"perfect-judge back-off vs imported precision@k:  max |gap| = {max(gaps):.2e}  (collapse anchor)")

# and the full verified harness:
CF._run_all()